# Practice 4. Neural networks for machine diagnostics: a bearing

Industrial AI, KBTU. Goes with Lectures 6–7 and `inclass_work_7.ipynb`.
**1.25 points.** Eleven tasks in two sections.

| section | tasks | data | what |
|---|---|---|---|
| 1 | 1–6 | CWRU, 16 time features | an MLP in PyTorch, learning curves, early stopping, against the Practice 3 baselines |
| 2 | 7–11 | CWRU, the raw signal | spectrum, spectrogram, band energies, a 1D CNN, an autoencoder, a fault at another load |

| Task | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 | 10 | 11 | total |
|---|---|---|---|---|---|---|---|---|---|---|---|---|
| code (tests) | 0.03 | 0.03 | 0.06 | 0.03 | 0.06 | 0.06 | 0.06 | 0.04 | 0.08 | 0.08 | 0.07 | 0.60 |
| answers | 0.05 | 0.05 | 0.04 | 0.05 | 0.04 | 0.10 | 0.04 | 0.06 | 0.07 | 0.07 | 0.08 | 0.65 |

Section 1 gives 0.60, Section 2 0.65.

**Data.** **CWRU bearings**, Case Western Reserve University Bearing Data Center, ten recordings
of the same motor (a healthy bearing and nine seeded faults). The notebook downloads them from
Kaggle by itself: <https://www.kaggle.com/datasets/brjapon/cwru-bearing-datasets>.

### How the tasks are built

Each task says what to write: the functions, their arguments and what they return, and sometimes
a variable with a result. How you write them is up to you, as long as you use the method the task
names. Under your cell sits a cell of **tests**. Run it after your code: it either prints
`all tests passed` or stops at the first check that failed and says what is wrong. The tests check
what your code does on small cases with known answers, not how it is written. For a network they
check what any correct network shows: the shapes, the output for given weights, a loss that falls
on a toy problem, never the exact numbers of a training run.

These tests are public. When you hand in, the same tests run again together with hidden ones, on
other inputs, and the share of tests passed gives the points for the code. Code that only fits the
public cases loses those points.

Cells marked *Common ground* are written for you. Run them as they are: they fix the windows, the
splits and the training loops, so that everybody's numbers can be compared.

The methods are yours: the per-class F1, the network and its training step, early stopping, the
spectrum and the band energies, the CNN, the autoencoder. Library versions are for checking your
result only.

### What you hand in

Two files:

1. this notebook, run top to bottom, named `Surname_Name_practice_4.ipynb`;
2. `practice_4_answers.md` with your answers to the questions under each task, named
   `Surname_Name_practice_4_answers.md`.

Every answer needs the numbers your notebook printed. A number the notebook does not print scores
zero, and so does a number printed with nothing said about it.

### The rules

- **The test part is touched only to measure**: in Tasks 6 and 9–11. Every choice (the epoch to stop at, the size of the network, the
  hyperparameters, the model to recommend) is made on the validation part.
- Every network is trained with a fixed seed, and the seed is printed next to its result.
- The notebook trains on a GPU if PyTorch finds one: the common cell sets `DEVICE`, and the data
  tensors live there. Put every network you build on it too, `make_mlp(...).to(DEVICE)`, and bring
  results back with `.cpu()` before NumPy or a plot. Your functions must work on both devices: the
  tests call them with CPU tensors. On a GPU the last digits of a training run differ from a CPU
  run with the same seed.
- Do not rename the functions and variables the tests use, and do not edit the test cells.
- Before handing in, restart the kernel and run all cells, then check your answers against the new
  output. The whole notebook runs in about fifteen minutes on a laptop CPU.
- Figures need axis labels and a title that says what to look at.
- Stuck? `inclass_work_7.ipynb`: Parts 1–6 do Section 1 on AI4I, Parts 12–14 do Section 2 on
  simulated signals.

---

# Section 1. A neural network that recognises the state of a bearing

Tasks 1–6, 0.60 points.

**The test rig.** A 2 hp electric motor drives a shaft through a torque sensor to a dynamometer
that loads it. The bearing under test sits at the drive end of the motor. Faults were seeded on
purpose by electro-discharge machining: a pit of 0.007, 0.014 or 0.021 inch (0.18, 0.36,
0.53 mm) on the **inner race** (IR), on a **ball** (B) or on the **outer race** (OR, at the
6 o'clock position). Two accelerometers recorded the vibration, one at the **drive end** (DE,
next to the faulty bearing) and one at the **fan end** (FE, the far end of the motor), 48 000
samples per second, about 10 s per recording.

**The ten states.** Motor load 1 hp, about 1772 rpm. One recording per state:

| state | `normal` | `B007` | `B014` | `B021` | `IR007` | `IR014` | `IR021` | `OR007` | `OR014` | `OR021` |
|---|---|---|---|---|---|---|---|---|---|---|
| fault | none | ball | ball | ball | inner race | inner race | inner race | outer race | outer race | outer race |
| size, inch | – | 0.007 | 0.014 | 0.021 | 0.007 | 0.014 | 0.021 | 0.007 | 0.014 | 0.021 |

The file names carry the record number at the end: `B007_1_123.mat` is record 123, state B007
at 1 hp, and its variables are `X123_DE_time`, `X123_FE_time` and `X123RPM`. The catalogue of
all records is on the data centre's site:
<https://engineering.case.edu/bearingdatacenter/48k-drive-end-bearing-fault-data>

**Why it matters.** Rolling bearings are among the parts that fail most often in motors, pumps
and fans. A condition monitoring system earns its price by telling a healthy bearing from a
damaged one, and where it is damaged, from a cheap accelerometer.

**The motor current.** The syllabus names vibration *and current* features. The CWRU rig
recorded no motor current, so this practice works with vibration only, from two sensors.

In [2]:
import warnings
warnings.filterwarnings("ignore")

import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

import sklearn
import torch
from torch import nn

BLUE, ORANGE, RED, GREEN, PURPLE, GREY = (
    "#2a6f8e", "#e07b39", "#c0392b", "#4a8f5a", "#9b3f9b", "#9aa0a6")

mpl.rcParams.update({
    "figure.dpi": 110,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.edgecolor": "#cccccc",
    "axes.grid": True,
    "axes.axisbelow": True,
    "grid.color": "#e6e6e6",
    "grid.linewidth": 0.8,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titlesize": 12,
    "axes.titleweight": "bold",
    "axes.titlelocation": "left",
    "axes.labelsize": 10,
    "font.size": 10,
    "font.family": ["Segoe UI", "DejaVu Sans", "sans-serif"],
    "legend.frameon": False,
    "lines.linewidth": 1.6,
})

pd.set_option("display.width", 130)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")

KAGGLE = {
    'cwru': 'brjapon/cwru-bearing-datasets',
}


def dataset_folder(name):
    """The folder with the files of a data set: the Kaggle copy (downloaded once, then cached),
    or data/<name>/ of the course repo if Kaggle is out of reach."""
    if KAGGLE[name]:
        try:
            import contextlib, io, logging, kagglehub
            logging.getLogger("kagglehub").setLevel(logging.ERROR)
            with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
                folder = Path(kagglehub.dataset_download(KAGGLE[name]))
            print(f"{name}: Kaggle {KAGGLE[name]}")
            return folder
        except Exception as exc:
            print(f"Kaggle unavailable ({type(exc).__name__}), looking for data/{name}/")
    for root in [Path.cwd(), *Path.cwd().parents]:
        if (root / "data" / name).is_dir():
            return root / "data" / name
    raise FileNotFoundError(f"no data/{name}/ here or above; run data/fetch.py in the course repo")

print(" | ".join(f"{m} {sys.modules[m].__version__}" for m in
                 ("numpy", "pandas", "matplotlib", "scipy", "sklearn", "torch") if m in sys.modules))

numpy 2.5.5 | pandas 3.1.1 | matplotlib 3.11.2 | scipy 1.18.1 | sklearn 1.9.1 | torch 2.8.0+cu128


In [3]:
import scipy.io as sio

CWRU = dataset_folder("cwru")            # raw/*.mat, and a ready-made feature table
MAT_FILES = sorted((CWRU / "raw").glob("*.mat"))
for path in MAT_FILES:
    m = sio.loadmat(path)
    shapes = {k: v.shape for k, v in m.items() if not k.startswith("__")}
    print(f"{path.name:22s}", shapes)

cwru: Kaggle brjapon/cwru-bearing-datasets
B007_1_123.mat         {'X123_DE_time': (487384, 1), 'X123_FE_time': (487384, 1), 'X123RPM': (1, 1)}
B014_1_190.mat         {'X190_DE_time': (486224, 1), 'X190_FE_time': (486224, 1), 'X190RPM': (1, 1)}
B021_1_227.mat         {'X227_DE_time': (486804, 1), 'X227_FE_time': (486804, 1), 'X227RPM': (1, 1)}
IR007_1_110.mat        {'X110_DE_time': (486224, 1), 'X110_FE_time': (486224, 1), 'X110RPM': (1, 1)}
IR014_1_175.mat        {'X217': (489125, 2), 'X217_DE_time': (489125, 1), 'X175_DE_time': (381890, 1), 'X175_FE_time': (381890, 1), 'X175RPM': (1, 1)}
IR021_1_214.mat        {'X214_DE_time': (485063, 1), 'X214_FE_time': (485063, 1), 'X214RPM': (1, 1)}
OR007_6_1_136.mat      {'X136_DE_time': (486804, 1), 'X136_FE_time': (486804, 1), 'X136RPM': (1, 1)}
OR014_6_1_202.mat      {'X202_DE_time': (484483, 1), 'X202_FE_time': (484483, 1), 'X202RPM': (1, 1)}
OR021_6_1_239.mat      {'X239_DE_time': (489125, 1), 'X239_FE_time': (489125, 1), 'X239RPM': (1, 1)

---

## Task 1. What is in the files  
*0.08 points: 0.03 for the code (tests), 0.05 for the answers*

Take inventory before you model anything. The cell above lists every variable in every file. All
recordings are sampled at $f_s = 48\,000$ Hz, so a recording of $n$ samples lasts $n / f_s$
seconds, and a window of 2048 samples lasts 42.7 ms.

**One file holds two recordings.** Find it. One of its recordings is the record named in the file
name; look the other one up in the CWRU catalogue (link above): which state, which motor load? The
ready-made feature table `feature_time_48k_2048_load_1.csv` in the same Kaggle folder
(`CWRU / "feature_time_48k_2048_load_1.csv"`) has a column `fault` and a column `max`. Its
`IR_014_1` windows were cut from one of the two recordings. Compare the `max` of its first
`IR_014_1` window with the largest value of the first 2048 samples of each recording to find out
which.

**Write**

- `de_signal(path)`: the drive-end signal of the record whose number ends the file name, a 1-D
  float array (`B007_1_123.mat` → `X123_DE_time`). It must work for any file named this way;
- `durations`: a dict `{file name: seconds}`, the length of `de_signal` of every file in
  `MAT_FILES` (keys are `path.name`);
- `shaft_rps`: a `float`, the shaft speed in revolutions per second: the mean of the RPM variables
  of the named records (the normal file has none), divided by 60;
- `window_revs`: a `float`, the shaft revolutions during one window of 2048 samples;
- `two_record_file`: a `str`, the name of the file that holds two recordings;
- `table_source`: a `str`, the name of the variable (such as `"X123_DE_time"`) the feature table
  took its `IR_014_1` windows from.

In [ ]:
def de_signal(path):
    ...


durations = ...
shaft_rps = ...
window_revs = ...
two_record_file = ...
table_source = ...

In [ ]:
# Tests for Task 1 (public). They check what your code does, not how it is written.
# Run this cell after your code; an AssertionError says which check failed.


def test_the_record_named_in_the_file():
    import tempfile
    with tempfile.TemporaryDirectory() as d:
        fake = Path(d) / "Fake_1_001.mat"
        variables = {"X001_DE_time": np.arange(5.0)[:, None],
                     "X001_FE_time": np.zeros((5, 1)),
                     "X217_DE_time": np.ones((3, 1))}
        sio.savemat(fake, variables)
        got = de_signal(fake)
    one_d = got is not None and np.asarray(got).shape == (5,)
    assert one_d, f"a 1-D array of the 5 samples of X001_DE_time, got {got}"
    assert np.allclose(got, np.arange(5.0)), "the record named in the file name, not another"


test_the_record_named_in_the_file()


def test_results_have_the_right_type():
    one_per_file = isinstance(durations, dict) and len(durations) == len(MAT_FILES)
    assert one_per_file, "durations: one entry per file"
    seconds = all(isinstance(v, float) and v > 0 for v in durations.values())
    assert seconds, "durations: seconds as floats"
    assert isinstance(shaft_rps, float) and isinstance(window_revs, float)
    names = [q.name for q in MAT_FILES]
    assert two_record_file in names, "two_record_file: the name of one of the files"
    is_name = isinstance(table_source, str) and table_source.endswith("_DE_time")
    assert is_name, "table_source: a variable name such as X123_DE_time"


test_results_have_the_right_type()


print("all tests passed")

**Questions** (answer them in `practice_4_answers.md`, Task 1, with the numbers your code printed):

1. How long is each recording, and which one is shorter than the rest? How many windows of 2048 samples does that leave it?
2. What is the shaft speed, and how long is one window in milliseconds and in shaft revolutions?
3. Which file holds two recordings, under which variables, and which of them is the record named in the file? What is the other one, by the catalogue? Which one did the feature table use (give the three maxima), and what would a model trained on that table have learned about "IR014"?
4. Plot 0.1 s of the drive-end signal of `normal`, `IR021`, `B021` and `OR021`, one panel each, the same vertical scale. Describe what you see.

In [ ]:
# your code for the questions: tables, figures, anything you quote in the answers

---

## Common ground for Tasks 2–6

Run this cell as it is. It cuts every recording into windows of 2048 samples, computes eight
features per window for each of the two sensors (16 in total), and splits the windows into three
parts so that everybody's numbers can be compared.

**The features.** For a window $x_1 \dots x_N$: root mean square
$\text{rms} = \sqrt{\tfrac{1}{N}\sum x_i^2}$, peak $\max |x_i|$, peak-to-peak, crest factor
peak / rms, shape factor rms / mean$|x|$, impulse factor peak / mean$|x|$, skewness, and kurtosis
$\tfrac{1}{N}\sum (x_i - \bar x)^4 / s^4 - 3$, which is 0 for a Gaussian signal. A bearing defect
is hit once per pass of a rolling element, and each hit is a short spike. A spike raises the
peak far more than the rms, and the fourth power in the kurtosis makes a few large samples
dominate the sum, so crest factor and kurtosis grow with impacts.

**The split is in time, inside every recording.** The first 60 % of the windows of each recording
go to *train*, the next 20 % to *validation*, the last 20 % to *test*. Two neighbouring windows
are 43 ms apart and look almost the same; a random split would put one of them in train and its
twin in test, and the test would measure memory.

The scaler is fitted on the train part only. `X_train` etc. are DataFrames for scikit-learn,
`Xt_train` etc. are scaled tensors for PyTorch, and the labels are the numbers 0–9 in the order
of `STATES`. `make_logreg()` and `make_forest()` are the two baselines of Practice 3, without the
class weights: here the classes are almost the same size. `predict(net, X)` gives a network's
predicted states as a NumPy array.

In [7]:
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# a GPU if PyTorch finds one
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("training on", DEVICE)

FS = 48_000           # samples per second
WIN = 2048            # samples per window (42.7 ms)
STATES = ["normal", "B007", "B014", "B021", "IR007", "IR014", "IR021",
          "OR007", "OR014", "OR021"]


def read_record(path):
    # (state, drive-end signal, fan-end signal) of the record named in the file name:
    # "OR007_6_1_136.mat" is record 136, state OR007;
    # "Time_Normal_1_098.mat" is the normal one
    parts = path.stem.split("_")
    record = parts[-1]
    m = sio.loadmat(path)
    state = "normal" if parts[0] == "Time" else parts[0]
    de = m[f"X{record}_DE_time"].ravel()
    fe = m[f"X{record}_FE_time"].ravel()
    return state, de, fe


def part_of(i, n):
    # window i of a recording with n windows:
    # the first 60 % train, the next 20 % val, the rest test
    if i < 0.6 * n:
        return "train"
    if i < 0.8 * n:
        return "val"
    return "test"


def window_features(x):
    rms = np.sqrt(np.mean(x ** 2))
    peak = np.abs(x).max()
    mean_abs = np.abs(x).mean()
    return {"rms": rms,
            "peak": peak,
            "p2p": x.max() - x.min(),
            "crest": peak / rms,
            "shape": rms / mean_abs,
            "impulse": peak / mean_abs,
            "skewness": stats.skew(x),
            "kurtosis": stats.kurtosis(x)}


rows = []
for path in MAT_FILES:
    state, de, fe = read_record(path)
    n = len(de) // WIN
    for i in range(n):
        row = {"state": state, "window": i, "part": part_of(i, n)}
        for sensor, x in (("DE", de), ("FE", fe)):
            for k, v in window_features(x[i * WIN:(i + 1) * WIN]).items():
                row[f"{sensor}_{k}"] = v
        rows.append(row)
feat = pd.DataFrame(rows)
FEATURES = [c for c in feat.columns if c[:3] in ("DE_", "FE_")]
feat["y"] = feat["state"].map(STATES.index)

# the three parts, as pandas (for scikit-learn) and as tensors (for PyTorch)
PARTS = ("train", "val", "test")
X_train, X_val, X_test = (feat.loc[feat["part"] == k, FEATURES] for k in PARTS)
y_train, y_val, y_test = (feat.loc[feat["part"] == k, "y"].to_numpy() for k in PARTS)
scaler = StandardScaler().fit(X_train)                 # fitted on train only


def to_tensor(X):
    return torch.tensor(scaler.transform(X), dtype=torch.float32).to(DEVICE)


Xt_train, Xt_val, Xt_test = (to_tensor(X) for X in (X_train, X_val, X_test))
yt_train, yt_val, yt_test = (torch.tensor(v).to(DEVICE) for v in (y_train, y_val, y_test))


def make_logreg():
    return make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000))


def make_forest(seed=0):
    return RandomForestClassifier(500, min_samples_leaf=3, random_state=seed, n_jobs=-1)


@torch.no_grad()
def predict(net, X):
    # the state with the largest output, for every row of X
    net.eval()
    return net(X).argmax(1).cpu().numpy()


print(feat.groupby("part", sort=False).size().to_string())
print(f"{len(FEATURES)} features: {FEATURES[:8]} and the same for FE")
counts = feat.pivot_table(index="state", columns="part", values="window", aggfunc="count")
print(counts.reindex(STATES)[["train", "val", "test"]].T.to_string())

training on cuda


part
train    1396
val       461
test      460
16 features: ['DE_rms', 'DE_peak', 'DE_p2p', 'DE_crest', 'DE_shape', 'DE_impulse', 'DE_skewness', 'DE_kurtosis'] and the same for FE
state  normal  B007  B014  B021  IR007  IR014  IR021  OR007  OR014  OR021
part                                                                     
train     142   143   143   143    143    112    142    143    142    143
val        47    47    47    47     47     37     47     47     47     48
test       47    47    47    47     47     37     47     47     47     47


---

## Task 2. The baseline from Practice 3  
*0.08 points: 0.03 for the code (tests), 0.05 for the answers*

In a confusion matrix $\mathbf{C}$ of $K$ classes, row $k$ is the true class $k$ and column $k$
the predicted class $k$. For each class, one against the rest:

$$TP_k = C_{kk} \qquad FP_k = \sum_i C_{ik} - TP_k \qquad FN_k = \sum_j C_{kj} - TP_k \qquad F_{1,k} = \frac{2\,TP_k}{2\,TP_k + FP_k + FN_k}$$

A false positive of class $k$ is a window of another class that landed in column $k$; a false
negative is a window of class $k$ that landed in another column. So the column sum minus the
diagonal gives $FP_k$, the row sum minus the diagonal $FN_k$. Macro-F1 is the plain mean of the
$K$ values: every class counts the same, however many windows it has. A class that is neither
present nor predicted has $2TP + FP + FN = 0$; count its $F_1$ as 0.

**Write**

- `per_class_f1(cm)`: for a $K \times K$ confusion matrix (array or nested list), the tuple
  `(f1, macro)`: a 1-D array of the $K$ values and their mean as a `float`;
- `logreg`, `forest`: `make_logreg()` and `make_forest()` fitted on the train part;
- `val_macro`: a dict `{"logistic": ..., "forest": ...}`, the macro-F1 of each on the validation
  part, from your `per_class_f1`.

In [ ]:
def per_class_f1(cm):
    ...


logreg = ...
forest = ...
val_macro = {"logistic": ..., "forest": ...}

In [ ]:
# Tests for Task 2 (public). They check what your code does, not how it is written.
# Run this cell after your code; an AssertionError says which check failed.


def test_three_classes_by_hand():
    got = per_class_f1([[5, 1, 0], [2, 3, 1], [0, 0, 4]])
    assert got is not None, "per_class_f1 returns (f1, macro)"
    f1, macro = got
    expected = [10 / 13, 6 / 10, 8 / 9]
    assert np.allclose(f1, expected), f"expected [0.769, 0.6, 0.889], got {f1}"
    assert np.isclose(macro, np.mean(expected)) and isinstance(macro, float)


test_three_classes_by_hand()


def test_absent_class_scores_zero():
    f1, macro = per_class_f1(np.array([[3, 0], [0, 0]]))
    no_nan = np.allclose(f1, [1, 0]) and np.isclose(macro, 0.5)
    assert no_nan, f"no NaN for an empty class, got {f1}"


test_absent_class_scores_zero()


def test_validation_scores_reported():
    assert set(val_macro) == {"logistic", "forest"}
    assert all(0 < v <= 1 for v in val_macro.values())


test_validation_scores_reported()


print("all tests passed")

**Questions** (answer them in `practice_4_answers.md`, Task 2, with the numbers your code printed):

1. On the validation part, give the accuracy and macro-F1 of both models and the F1 of every state.
2. Plot the validation confusion matrix of the better model with the state names on the axes. Which states are recognised perfectly, and which are confused with which? Is the confusion between fault *locations* (ball against inner race) or between *sizes* of the same fault?
3. Why macro-F1 and not only accuracy here, although the classes are almost the same size?

In [ ]:
# your code for the questions: tables, figures, anything you quote in the answers

---

## Task 3. The network  
*0.10 points: 0.06 for the code (tests), 0.04 for the answers*

A multilayer perceptron is a chain of linear layers with a non-linear function between them:

$$\mathbf{h}_1 = \text{ReLU}(\mathbf{W}_1 \mathbf{x} + \mathbf{b}_1), \qquad \mathbf{h}_2 = \text{ReLU}(\mathbf{W}_2 \mathbf{h}_1 + \mathbf{b}_2), \qquad \mathbf{z} = \mathbf{W}_3 \mathbf{h}_2 + \mathbf{b}_3$$

$\text{ReLU}(v) = \max(0, v)$, applied to every component. The output $\mathbf{z}$ has one number
per state (a *logit*), and the predicted state is the largest. The cross-entropy turns the logits
into probabilities with the softmax, $p_k = e^{z_k} / \sum_j e^{z_j}$, and charges $-\log p_y$ for
the true state $y$. Without the ReLUs the three layers would collapse into one matrix,
$\mathbf{W}_3 \mathbf{W}_2 \mathbf{W}_1$, and the network would be a logistic regression again.

Training repeats four steps per mini-batch, exactly as in In-class work 7: clear the gradients,
forward pass and loss, backward pass, optimiser step.

**Write**

- `make_mlp(n_in, hidden, n_out)`: an `nn.Sequential` of `Linear` and `ReLU` layers: for each size
  `h` in the list `hidden` a `Linear` to `h` followed by a `ReLU`, then a `Linear` to `n_out` with
  no activation;
- `train_one_epoch(net, loader, loss_fn, opt)`: one pass over the mini-batches of `loader`, one
  optimiser step per batch;
- `evaluate(net, X, y, loss_fn)`: `(loss, accuracy)` as Python floats on the whole of `X`, `y`, in
  evaluation mode and without tracking gradients;
- `net`, `hist`: the network of this practice trained for **300 epochs**: `torch.manual_seed(0)`
  right before `make_mlp(16, [64, 64], 10).to(DEVICE)`, `nn.CrossEntropyLoss()`,
  `torch.optim.Adam(..., lr=1e-3)`, mini-batches of 64 from the train part shuffled with
  `generator=torch.Generator().manual_seed(0)`. `hist` is a `pd.DataFrame`, one row per epoch, with
  columns `"train_loss"` (whole train part), `"val_loss"` and `"val_acc"` (validation part), from
  `evaluate` after each epoch.

In [ ]:
from torch.utils.data import TensorDataset, DataLoader


def make_mlp(n_in, hidden, n_out):
    ...


def train_one_epoch(net, loader, loss_fn, opt):
    ...


def evaluate(net, X, y, loss_fn):
    ...


net = ...
hist = ...

In [ ]:
# Tests for Task 3 (public). They check what your code does, not how it is written.
# Run this cell after your code; an AssertionError says which check failed.


def test_layers_and_weights():
    m = make_mlp(16, [64, 64], 10)
    assert isinstance(m, nn.Sequential), "make_mlp returns an nn.Sequential"
    assert [type(l).__name__ for l in m] == ["Linear", "ReLU", "Linear", "ReLU", "Linear"]
    n_weights = sum(q.numel() for q in m.parameters())
    assert n_weights == 5898, "16*64+64 + 64*64+64 + 64*10+10 weights"
    assert tuple(m(torch.zeros(7, 16)).shape) == (7, 10)


test_layers_and_weights()


def test_forward_pass_on_fixed_weights():
    m = make_mlp(2, [2], 1)
    lin = [l for l in m if isinstance(l, nn.Linear)]
    with torch.no_grad():
        lin[0].weight.copy_(torch.tensor([[1.0, -1.0], [-1.0, 1.0]]))
        lin[0].bias.zero_()
        lin[1].weight.copy_(torch.tensor([[1.0, 2.0]]))
        lin[1].bias.fill_(0.5)
        out = m(torch.tensor([[3.0, 1.0], [1.0, 3.0]])).ravel()
    assert torch.allclose(out, torch.tensor([2.5, 4.5])), (
        f"ReLU([2, -2]) = [2, 0] gives 2.5; ReLU([-2, 2]) gives 4.5; got {out}")


test_forward_pass_on_fixed_weights()


def test_one_step_per_batch_gradients_cleared():
    lin = nn.Linear(1, 1, bias=False)
    with torch.no_grad():
        lin.weight.zero_()
    loader = DataLoader(TensorDataset(torch.ones(1, 1), torch.ones(1, 1)), batch_size=1)
    opt = torch.optim.SGD(lin.parameters(), lr=0.5)
    for _ in range(2):
        train_one_epoch(lin, loader, nn.MSELoss(), opt)
    w = lin.weight.item()
    assert abs(w - 1.0) < 1e-6, (
        f"one SGD step from w = 0 gives w = 1, and the second epoch must not move it; "
        f"got {w}. Are the gradients cleared before each batch?")


test_one_step_per_batch_gradients_cleared()


def test_evaluate_on_fixed_outputs():
    lin = nn.Linear(2, 2)
    with torch.no_grad():
        lin.weight.copy_(torch.eye(2))
        lin.bias.zero_()
    X = torch.tensor([[2.0, 0.0], [0.0, 1.0], [1.0, 3.0]])
    yy = torch.tensor([0, 0, 1])
    got = evaluate(lin, X, yy, nn.CrossEntropyLoss())
    assert got is not None and len(got) == 2, "evaluate returns (loss, accuracy)"
    want = nn.functional.cross_entropy(X, yy).item()
    loss_ok = isinstance(got[0], float) and abs(got[0] - want) < 1e-6
    assert loss_ok, f"loss: expected {want:.4f}, got {got[0]}"
    assert abs(got[1] - 2 / 3) < 1e-6, f"two of three rows right, got {got[1]}"


test_evaluate_on_fixed_outputs()


def test_the_network_learns_a_toy_problem():
    g = torch.Generator().manual_seed(0)
    Xd = torch.randn(400, 2, generator=g)
    # three classes that no straight line separates
    yd = (Xd[:, 0] > 0).long() + (Xd[:, 1] > 0).long()
    torch.manual_seed(0)
    m = make_mlp(2, [32], 3)
    shuffle_rng = torch.Generator().manual_seed(0)
    loader = DataLoader(TensorDataset(Xd, yd), batch_size=32, shuffle=True,
                        generator=shuffle_rng)
    opt = torch.optim.Adam(m.parameters(), lr=0.01)
    before = evaluate(m, Xd, yd, nn.CrossEntropyLoss())
    for _ in range(100):
        train_one_epoch(m, loader, nn.CrossEntropyLoss(), opt)
    after = evaluate(m, Xd, yd, nn.CrossEntropyLoss())
    learned = after[0] < 0.5 * before[0] and after[1] > 0.95
    assert learned, f"loss {before[0]:.3f} -> {after[0]:.3f}, accuracy {after[1]:.3f}"


test_the_network_learns_a_toy_problem()


def test_history_recorded():
    one_row_per_epoch = isinstance(hist, pd.DataFrame) and len(hist) == 300
    assert one_row_per_epoch, "hist: one row per epoch, 300 epochs"
    assert {"train_loss", "val_loss", "val_acc"} <= set(hist.columns)


test_history_recorded()


print("all tests passed")

**Questions** (answer them in `practice_4_answers.md`, Task 3, with the numbers your code printed):

1. How many weights does the network have? Count them by hand from the layer sizes and check with `sum(p.numel() for p in net.parameters())`. How does that compare with the number of training windows?
2. Print the three numbers of `hist` at epochs 0, 10, 50, 100, 200 and 299.
3. Where is the validation loss lowest, and what is the validation accuracy there and at epoch 299?

In [ ]:
# your code for the questions: tables, figures, anything you quote in the answers

---

## Task 4. Learning curves  
*0.08 points: 0.03 for the code (tests), 0.05 for the answers*

A training run is one curve, and one curve hides the size of the network it came from. Wrap the
recipe of Task 3 into a function, so that every later run is one line.

The two losses tell different stories. The training loss falls as long as the network can fit the
train windows better, which a large enough network always can. The validation loss falls only
while what it learns also holds for unseen windows. The gap between them is the part learned by
heart.

**Write**

- `run(hidden, epochs=300, seed=0)`: `(net, hist)` with the recipe of Task 3 for the hidden sizes
  `hidden`: `torch.manual_seed(seed)` before `make_mlp(16, hidden, 10).to(DEVICE)`, the loader
  shuffled with `torch.Generator().manual_seed(seed)`, Adam with `lr=1e-3`, `hist` with the same
  three columns. The same seed gives the same run;
- `curves`: a dict with the `hist` of three runs of 300 epochs, seed 0, keys `"[4]"`, `"[64, 64]"`
  and `"[256, 256]"` (`str(hidden)`).

In [ ]:
def run(hidden, epochs=300, seed=0):
    ...


curves = {"[4]": ..., "[64, 64]": ..., "[256, 256]": ...}

In [ ]:
# Tests for Task 4 (public). They check what your code does, not how it is written.
# Run this cell after your code; an AssertionError says which check failed.


def test_a_short_run():
    got = run([8], epochs=3, seed=1)
    assert got is not None and len(got) == 2, "run returns (net, hist)"
    m, h = got
    assert isinstance(h, pd.DataFrame) and len(h) == 3
    assert {"train_loss", "val_loss", "val_acc"} <= set(h.columns)
    n_weights = sum(q.numel() for q in m.parameters())
    assert n_weights == 16 * 8 + 8 + 8 * 10 + 10, "make_mlp(16, [8], 10)"


test_a_short_run()


def test_the_seed_fixes_the_run():
    a = run([8], epochs=2, seed=1)[1]
    b = run([8], epochs=2, seed=1)[1]
    c = run([8], epochs=2, seed=2)[1]
    same = np.allclose(a["val_loss"], b["val_loss"], atol=1e-6)
    assert same, "the same seed must give the same run"
    differs = not np.allclose(a["val_loss"], c["val_loss"])
    assert differs, "another seed must give another run"


test_the_seed_fixes_the_run()


def test_three_curves():
    assert set(curves) == {"[4]", "[64, 64]", "[256, 256]"}
    assert all(len(h) == 300 for h in curves.values())


test_three_curves()


print("all tests passed")

**Questions** (answer them in `practice_4_answers.md`, Task 4, with the numbers your code printed):

1. Plot the learning curves of `[64, 64]`: train and validation loss on one axis, validation accuracy on a second panel, the epoch of the lowest validation loss marked. Describe the three stages you see, with epochs. What happens to the gap between the two losses?
2. The validation loss at epoch 299 is much worse than at its minimum, yet the validation accuracy hardly moved. How can both be true? (Think about what cross-entropy punishes that accuracy does not see.)
3. Plot the validation losses of the three sizes. Which one underfits, which one overfits fastest? Give the lowest validation loss of each and its epoch.

In [ ]:
# your code for the questions: tables, figures, anything you quote in the answers

---

## Task 5. Early stopping  
*0.10 points: 0.06 for the code (tests), 0.04 for the answers*

Early stopping watches the validation loss after every epoch. It remembers the best value and
the weights that gave it. When the loss has not improved for `patience` epochs in a row, training
stops and the remembered weights are put back. An epoch improves only with a loss strictly below
the best so far.

`net.state_dict()` returns the network's *live* tensors: they keep changing as training goes on.
To keep the weights of the best epoch you need `copy.deepcopy(net.state_dict())`.

**Write**

- a class `EarlyStopper(patience)` with the attributes `best_loss` (start `inf`), `best_epoch`,
  `best_state` (a copy of the best weights) and `epoch` (the number of the last epoch seen, from 0),
  and a method `step(val_loss, net)`, called once per epoch after evaluating, that returns `True`
  when `patience` epochs have passed since the best one;
- `run_early(hidden, seed=0, patience=20, max_epochs=500)`: `(net, hist, stopper)`: the recipe of
  `run` with your `EarlyStopper`, stopping when `step` says so, the best weights loaded back into
  `net` before it is returned;
- `net_es`, `hist_es`, `stopper_es`: `run_early([64, 64])`.

In [ ]:
import copy


class EarlyStopper:
    def __init__(self, patience):
        ...

    def step(self, val_loss, net):
        ...


def run_early(hidden, seed=0, patience=20, max_epochs=500):
    ...


net_es, hist_es, stopper_es = ..., ..., ...

In [ ]:
# Tests for Task 5 (public). They check what your code does, not how it is written.
# Run this cell after your code; an AssertionError says which check failed.


def test_a_scripted_curve():
    m = nn.Linear(1, 1)
    st = EarlyStopper(patience=3)
    losses = [1.0, 0.8, 0.7, 0.72, 0.69, 0.71, 0.73, 0.74, 0.75, 0.70, 0.9]
    for ep, l in enumerate(losses):
        with torch.no_grad():
            m.weight.fill_(ep)                 # the weight records the epoch
        if st.step(l, m):
            break
    got = (ep, st.best_epoch, st.best_loss, st.best_state["weight"].item())
    assert got == (7, 4, 0.69, 4.0), (
        f"(stopped at, best epoch, best loss, saved weight) should be (7, 4, 0.69, 4.0), "
        f"got {got}; a saved weight of 7.0 means the state was not copied")


test_a_scripted_curve()


def test_equal_loss_is_no_improvement():
    st = EarlyStopper(patience=2)
    stops = [st.step(1.0, nn.Linear(1, 1)) for _ in range(3)]
    stopped_right = stops == [False, False, True] and st.best_epoch == 0
    assert stopped_right, f"a loss equal to the best does not count, got {stops}"


test_equal_loss_is_no_improvement()


def test_the_run_is_reported():
    assert isinstance(stopper_es, EarlyStopper) and isinstance(hist_es, pd.DataFrame)
    assert len(hist_es) == stopper_es.epoch + 1, "one row of hist_es per epoch trained"


test_the_run_is_reported()


print("all tests passed")

**Questions** (answer them in `practice_4_answers.md`, Task 5, with the numbers your code printed):

1. Report the epoch it stopped at, the best epoch, and the validation loss and accuracy of the restored network. Compare with Task 3: the best epoch there, and the network at epoch 299. Plot the curves of this run with both epochs marked.
2. Why must the saved weights be a *copy*? What would `self.best_state = net.state_dict()` give you after training, and which public test catches it?
3. Early stopping chooses the epoch on the validation part. Is the validation accuracy of the restored network an honest estimate for new data? Why does Task 6 use the test part?

In [ ]:
# your code for the questions: tables, figures, anything you quote in the answers

---

## Task 6. The network against the baseline, on the test part  
*0.16 points: 0.06 for the code (tests), 0.10 for the answers*

Now open the test part. One network is one draw: its starting weights and the order of its
mini-batches come from the seed. Before you call one model better than another, see how far each
moves when only the seed changes.

**Write**

- `test_table`: a `pd.DataFrame` indexed by `"logistic"`, `"forest"`, `"network"` with columns
  `"accuracy"` and `"macro-F1"` on the **test** part: `logreg` and `forest` of Task 2, `net_es` of
  Task 5;
- `spread`: a dict `{"forest": [...], "network": [...]}`, the test macro-F1 of `make_forest(seed)`
  fitted on the train part and of `run_early([64, 64], seed=seed)`, for seeds 0, 1, 2, 3, 4 in this
  order.

In [ ]:
test_table = ...
spread = {"forest": ..., "network": ...}

In [ ]:
# Tests for Task 6 (public). They check what your code does, not how it is written.
# Run this cell after your code; an AssertionError says which check failed.


def test_table_and_spread():
    assert isinstance(test_table, pd.DataFrame)
    assert list(test_table.index) == ["logistic", "forest", "network"]
    assert {"accuracy", "macro-F1"} <= set(test_table.columns)
    assert set(spread) == {"forest", "network"}, "five seeds each"
    assert all(len(v) == 5 for v in spread.values()), "five seeds each"


test_table_and_spread()


print("all tests passed")

**Questions** (answer them in `practice_4_answers.md`, Task 6, with the numbers your code printed):

1. One table on the test part: accuracy and macro-F1 of the logistic regression, the forest and the early-stopped network.
2. Give the mean, the smallest and the largest test macro-F1 over the five seeds of each model, and the training time of each.
3. Plot the test confusion matrix of the seed-0 network. Which states does it confuse? Are they the same as for the forest in Task 2?
4. Is the network better than the forest? Answer with the spread over seeds, not with one pair of numbers. Then say which model you would install on a motor, and why. Training time, the number of weights and what a maintenance engineer can check are fair arguments.
5. Ball faults are the hard ones. One possible physical reason: a ball spins and rolls, so its defect does not touch the races at every pass. Look at your Task 1 figure for B021: does it look more like `normal` or like `IR021`?

In [ ]:
# your code for the questions: tables, figures, anything you quote in the answers

---

# Section 2. From the raw signal: spectrum, CNN, autoencoder

Tasks 7–11, 0.65 points. Section 1 trained a network on 16 hand-made features. Here the network
gets the spectrum and then the raw signal itself.

**The data** is the CWRU recording set of Section 1: one 2 hp motor, a healthy drive-end bearing
and nine seeded faults (ball, inner race, outer race; 0.007, 0.014 and 0.021 inch), motor load
1 hp, about 1772 rpm, 48 000 samples per second. Section 2 uses only the **drive-end**
accelerometer, next to the faulty bearing, and the same split in time inside every recording:
the first 60 % of the windows train, the next 20 % validation, the last 20 % test.

**The bearing.** The data centre gives the defect frequencies of the drive-end bearing
(6205-2RS JEM SKF) as multiples of the shaft speed: inner race (BPFI) 5.4152, outer race (BPFO)
3.5848, rolling element 4.7135. A defect is struck that many times per revolution, so at
29.5 rev/s an outer race defect is struck about 106 times a second.
<https://engineering.case.edu/bearingdatacenter/bearing-information>

**One more recording.** The file `IR014_1_175.mat` also holds record 217, IR021 at a motor load
of **3 hp**, about 1730 rpm (Task 1). No model here trains on it. Task 11 uses it as a fault the
models have seen, at another load.

### Common ground for Tasks 7–11

Run as it is. `raw` holds every window of the drive-end signal, `(windows, 2048)`; `y` the state
number of every window and `yt` the same as a tensor on `DEVICE`; `tr`, `va`, `te` boolean masks
of the three parts; `signals[state]` the whole recording; `raw_217` the windows of the 3 hp
recording. It also gives you `fit_early` (the training loop of Section 1 with early stopping) and
`f1_by_state`, the F1 of every state and the macro-F1.

In [23]:
import copy
import time
from scipy.signal import hilbert, spectrogram, find_peaks
from sklearn.metrics import f1_score

F_SHAFT = 1772 / 60                                  # Hz, motor load 1 hp
# defect frequencies of the drive-end bearing 6205-2RS JEM SKF,
# as multiples of the shaft speed
MULT = {"BPFI, inner race": 5.4152, "BPFO, outer race": 3.5848, "rolling element": 4.7135}

signals = {}
rows = []
for path in MAT_FILES:
    state, de, _ = read_record(path)                  # drive end only in Section 2
    signals[state] = de
    n = len(de) // WIN
    for i in range(n):
        rows.append({"state": state, "y": STATES.index(state), "window": i,
                     "part": part_of(i, n)})
meta = pd.DataFrame(rows)

window_list = [signals[s][i * WIN:(i + 1) * WIN]
               for s, i in zip(meta["state"], meta["window"])]
raw = np.stack(window_list).astype(np.float32)
y = meta["y"].to_numpy()
yt = torch.tensor(y).to(DEVICE)
tr, va, te = (meta["part"].eq(k).to_numpy() for k in ("train", "val", "test"))

# record 217 sits in the IR014 file next to record 175 (Task 1): IR021 at 3 hp
ir014_file = next(q for q in MAT_FILES if q.stem.endswith("_175"))
de_217 = sio.loadmat(ir014_file)["X217_DE_time"].ravel()
windows_217 = [de_217[i * WIN:(i + 1) * WIN] for i in range(len(de_217) // WIN)]
raw_217 = np.stack(windows_217).astype(np.float32)


def f1_by_state(y_true, pred):
    # (F1 of each of the ten states, their plain mean = macro-F1)
    f1 = f1_score(y_true, pred, labels=range(len(STATES)), average=None, zero_division=0)
    return f1, f1.mean()


def fit_early(net, X_tr, y_tr, X_va, y_va, seed=0, lr=1e-3, batch=64, patience=20,
              max_epochs=300):
    # the training loop of Section 1 with early stopping on the validation loss;
    # returns the network with the weights of its best epoch, and that epoch
    loss_fn = nn.CrossEntropyLoss()
    opt = torch.optim.Adam(net.parameters(), lr=lr)
    g = torch.Generator().manual_seed(seed)
    best_loss = np.inf
    best_state = None
    best_epoch = -1
    for epoch in range(max_epochs):
        net.train()
        for i in torch.randperm(len(X_tr), generator=g).split(batch):
            opt.zero_grad()
            loss_fn(net(X_tr[i]), y_tr[i]).backward()
            opt.step()

        net.eval()
        with torch.no_grad():
            v = loss_fn(net(X_va), y_va).item()
        if v < best_loss:
            best_loss = v
            best_state = copy.deepcopy(net.state_dict())
            best_epoch = epoch
        if epoch - best_epoch >= patience:
            break
    net.load_state_dict(best_state)
    return net, best_epoch


print(f"{len(y)} windows of {WIN} samples: "
      f"train {tr.sum()}, val {va.sum()}, test {te.sum()}")
print(f"the 3 hp recording (IR021 at 3 hp): {len(raw_217)} windows")
defect_hz = {k: round(v * F_SHAFT, 1) for k, v in MULT.items()}
print("defect frequencies at 1772 rpm, Hz:", defect_hz)

2317 windows of 2048 samples: train 1396, val 461, test 460
the 3 hp recording (IR021 at 3 hp): 238 windows
defect frequencies at 1772 rpm, Hz: {'BPFI, inner race': 159.9, 'BPFO, outer race': 105.9, 'rolling element': 139.2}


---

## Task 7. The spectrum and the defect frequencies  
*0.10 points: 0.06 for the code (tests), 0.04 for the answers*

`np.fft.rfft(x)` returns one complex number per frequency from 0 to $f_s/2$:
$X_k = \sum_n x_n e^{-2\pi i k n / n_x}$ for a signal of $n_x$ samples. A sine
$A \sin(2\pi f n / f_s)$ is the sum of two complex exponentials of amplitude $A/2$, one at $+f$
and one at $-f$. Over a whole number of periods every term of the sum for the bin at $+f$ adds
up in phase, so that bin has magnitude $A\, n_x / 2$, and `rfft` keeps only the positive half. So
the amplitude spectrum is

$$a_k = \frac{2}{n_x}\,\big|X_k\big|, \qquad f_k = k\,\frac{f_s}{n_x}.$$

**The envelope.** The strikes of a defect ring the housing at a high frequency; how *often* they
come is in the outline of the signal. The envelope is $e = |\text{hilbert}(x - \bar x)|$
(`scipy.signal.hilbert`); its spectrum, after its own mean is subtracted, has lines at the strike
rate and its multiples.

**Write**

- `amplitude_spectrum(x, fs)`: `(freqs, amp)`, the $f_k$ and the $a_k$ above, 1-D arrays;
- `envelope_peak(x, fs, lo=20, hi=500)`: the frequency in Hz of the strongest line of the envelope
  spectrum with $\text{lo} < f < \text{hi}$, a `float`;
- `peak_hz`: a dict with `envelope_peak` of the first second (`signals[state][:FS]`) of `"normal"`,
  `"IR021"`, `"OR021"` and `"B021"`.

In [ ]:
def amplitude_spectrum(x, fs):
    ...


def envelope_peak(x, fs, lo=20, hi=500):
    ...


peak_hz = {"normal": ..., "IR021": ..., "OR021": ..., "B021": ...}

In [ ]:
# Tests for Task 7 (public). They check what your code does, not how it is written.
# Run this cell after your code; an AssertionError says which check failed.


def test_two_sines():
    tt = np.arange(8000) / 8000
    x = 0.7 * np.sin(2 * np.pi * 50 * tt) + 0.2 * np.sin(2 * np.pi * 1200 * tt)
    got = amplitude_spectrum(x, 8000)
    assert got is not None, "amplitude_spectrum returns (freqs, amp)"
    f, a = got
    top = np.argsort(a)[-2:][::-1]
    right_f = np.allclose(f[top], [50, 1200])
    assert right_f and np.allclose(a[top], [0.7, 0.2], atol=1e-6), (
        f"peaks at 50 and 1200 Hz with amplitudes 0.7 and 0.2, got {f[top]} and {a[top]}")


test_two_sines()


def test_envelope_of_a_modulated_carrier():
    tt = np.arange(8000) / 8000
    x = (1 + 0.5 * np.cos(2 * np.pi * 100 * tt)) * np.sin(2 * np.pi * 3000 * tt)
    got = envelope_peak(x, 8000)
    assert got is not None and abs(got - 100) < 1, (
        f"a 3 kHz carrier pulsing 100 times a second: "
        f"the envelope line is at 100 Hz, got {got}")


test_envelope_of_a_modulated_carrier()


def test_peaks_reported():
    assert set(peak_hz) == {"normal", "IR021", "OR021", "B021"}
    assert all(20 < v < 500 for v in peak_hz.values())


test_peaks_reported()


print("all tests passed")

**Questions** (answer them in `practice_4_answers.md`, Task 7, with the numbers your code printed):

1. Plot the amplitude spectra of the first second of `normal`, `IR021`, `OR021` and `B021` up to 6 kHz, logarithmic vertical axis. Where does the energy of the faulty bearings sit compared with the healthy one?
2. Plot their envelope spectra from 0 to 500 Hz with the shaft speed and the three defect frequencies marked. Give the strongest envelope line of each state. Which of them sits on its own defect frequency? Which does not, and what does that tell you about the ball fault?

In [ ]:
# your code for the questions: tables, figures, anything you quote in the answers

---

## Task 8. The spectrogram  
*0.10 points: 0.04 for the code (tests), 0.06 for the answers*

A spectrogram cuts the signal into short overlapping slices and takes the spectrum of each. With
`spectrogram(x, fs, nperseg=128, noverlap=96)` a slice is 128 samples (2.7 ms at 48 kHz), slices
start 32 samples (0.67 ms) apart, and the frequency bins are 375 Hz apart. A strike rings for a
millisecond or two, so it shows up as a vertical stripe.

To measure the time between strikes, sum the power of each slice over a band where they ring,
take the natural logarithm, and find the peaks of that curve with
`find_peaks(log_energy, prominence=0.5, distance=4)`.

**Write**

- `strike_gaps(x, fs, f_lo=2000, f_hi=6000)`: a 1-D array of the times in **milliseconds** between
  neighbouring strikes, measured as above with the band $f_{lo} < f < f_{hi}$;
- `gap_ms`: a dict `{"IR021": ..., "OR021": ...}`, the median of `strike_gaps` over the first
  0.1 s of each recording.

In [ ]:
def strike_gaps(x, fs, f_lo=2000, f_hi=6000):
    ...


gap_ms = {"IR021": ..., "OR021": ...}

In [ ]:
# Tests for Task 8 (public). They check what your code does, not how it is written.
# Run this cell after your code; an AssertionError says which check failed.


def test_a_burst_every_ten_ms():
    rng_ = np.random.default_rng(0)
    fs = 48_000
    tt = np.arange(fs // 10) / fs
    # a steady hum, then bursts
    x = 0.01 * np.sin(2 * np.pi * 2500 * tt) + 0.001 * rng_.normal(size=fs // 10)
    burst = np.sin(2 * np.pi * 3000 * np.arange(48) / fs) * np.exp(-np.arange(48) / 12)
    for k in range(10):
        x[k * 480 + 100: k * 480 + 148] += burst
    got = strike_gaps(x, fs)
    assert got is not None and len(got) >= 7, f"ten bursts give nine gaps, got {got}"
    assert abs(np.median(got) - 10) < 1, (
        f"the bursts are 10 ms apart, median gap {np.median(got)}")


test_a_burst_every_ten_ms()


def test_gaps_reported():
    assert set(gap_ms) == {"IR021", "OR021"}
    assert all(1 < v < 40 for v in gap_ms.values())


test_gaps_reported()


print("all tests passed")

**Questions** (answer them in `practice_4_answers.md`, Task 8, with the numbers your code printed):

1. Plot the spectrograms of the first 0.1 s of `normal`, `IR021` and `OR021`, power in decibels (`10 * np.log10(S + 1e-12)`), one colour scale for all three, up to 24 kHz. In which frequency band do the strikes show up?
2. How many strikes in 0.1 s and what median gap do you measure for OR021 and IR021? Compare with one period of the BPFO and of the BPFI.
3. In IR021 the stripes are not all equally strong. Why would an inner race defect be struck harder at some moments? (Which part of the bearing turns with the shaft?)

In [ ]:
# your code for the questions: tables, figures, anything you quote in the answers

---

## Task 9. Spectral features and the network of Section 1  
*0.15 points: 0.08 for the code (tests), 0.07 for the answers*

Section 1 described a window by eight numbers in time. Here we describe it by where its energy
sits in frequency: split the power spectrum $|X_k|^2$ into $B$ equal bands and take the logarithm
of the energy in each,

$$e_b = \log_{10} \Big(\sum_{k \in \text{band } b} |X_k|^2 + 10^{-12}\Big).$$

By Parseval's theorem, $\sum_n x_n^2 = \frac{1}{N}\sum_k |X_k|^2$ over the full spectrum, so the
band energies split the energy of the window by frequency; up to the taper of the window, the
squared rms of Section 1 is proportional to their sum. Multiply the window by a Hann window
(`np.hanning(len(x))`) before the transform: it tapers the ends to zero, so that cutting the
signal at the edges does not smear a sharp line over its neighbours. Leave out the DC bin $k = 0$
and split the remaining bins of `rfft` into $B$ nearly equal parts with `np.array_split`. The
logarithm turns a range of a million into a range of six, which a network handles far better.

**Write**

- `band_energies(x, n_bands=32)`: the 1-D array of the $e_b$;
- `B`, `B_217`: the band energies of every window of `raw` and of `raw_217`, arrays `(windows, 32)`;
- `Bt`, `Bt_217`: the same scaled by a `StandardScaler` fitted on the train rows of `B`, as
  float32 tensors on `DEVICE`. Task 11 needs them;
- `forest_b`: `RandomForestClassifier(500, min_samples_leaf=3, random_state=0, n_jobs=-1)` fitted
  on `B[tr]`; `mlp_b`: `make_mlp(32, [64, 64], 10)` with `torch.manual_seed(0)` before it, trained
  with `fit_early(..., lr=3e-3)` on `Bt`;
- `bands_f1`: a dict `{"forest": ..., "network": ...}`, their test macro-F1.

In [ ]:
def band_energies(x, n_bands=32):
    ...


B, B_217 = ..., ...
Bt, Bt_217 = ..., ...
forest_b = ...
mlp_b = ...
bands_f1 = {"forest": ..., "network": ...}

In [ ]:
# Tests for Task 9 (public). They check what your code does, not how it is written.
# Run this cell after your code; an AssertionError says which check failed.


def test_a_sine_lands_in_its_band():
    sine = np.sin(2 * np.pi * 3300 * np.arange(2048) / 48_000)
    b = band_energies(sine)
    assert b is not None and np.asarray(b).shape == (32,), "32 bands"
    b = np.sort(np.asarray(b))
    assert np.argmax(band_energies(sine)) == 4, "bands of 750 Hz: 3.3 kHz is in band 4"
    assert b[-1] - b[-2] > 1, (
        "the Hann window keeps the line out of the other bands: more than a decade ahead")


test_a_sine_lands_in_its_band()


def test_energies_add_up():
    x = np.random.default_rng(1).normal(size=2048)
    e = band_energies(x, n_bands=8)
    power = np.abs(np.fft.rfft(x * np.hanning(2048))) ** 2
    total = np.sum(10 ** np.asarray(e))
    assert np.isclose(total, power[1:].sum(), rtol=1e-6), (
        "the bands share the power without the DC bin")


test_energies_add_up()


def test_features_and_scores_reported():
    assert np.asarray(B).shape == (len(raw), 32)
    assert np.asarray(B_217).shape == (len(raw_217), 32)
    assert set(bands_f1) == {"forest", "network"}
    assert all(0 < v <= 1 for v in bands_f1.values())


test_features_and_scores_reported()


print("all tests passed")

**Questions** (answer them in `practice_4_answers.md`, Task 9, with the numbers your code printed):

1. Test macro-F1 of the forest and of the network on the band energies, and the F1 of every state for the network.
2. Compare with Section 1, where 16 time-domain features from **two** sensors gave a test macro-F1 of about 0.92 with the ball faults as the weak states. What happened to the ball faults, and why do the bands carry more?
3. A result this good on CWRU should make you suspicious. Each state is one recording. What could the band energies have learned besides the fault? (Think about everything that differs between two recordings made on different days.)

In [ ]:
# your code for the questions: tables, figures, anything you quote in the answers

---

## Task 10. A CNN on the raw signal  
*0.15 points: 0.08 for the code (tests), 0.07 for the answers*

A one-dimensional convolution slides a short filter $\mathbf{w} \in \mathbb{R}^{k}$ along the signal
and gives one number per position $j$:

$$y_j = \sum_{u=0}^{k-1} w_u\, x_{sj + u} + b$$

with the stride $s$. This is the dot product of the filter with a stretch of the signal, so it is
large where the signal looks like the filter: a filter shaped like a short burst at 3 kHz answers
loudly wherever such a burst rings. The same weights are used at every position, so a layer
`Conv1d(c_in, c_out, k)` has $c_{out}(c_{in} k + 1)$ weights however long the signal is.
`AdaptiveMaxPool1d(1)` keeps, for each filter, its strongest answer anywhere in the window: a
strike counts wherever it falls.

**Write**

- `make_cnn()`: an `nn.Sequential` with, in this order, `Conv1d(1, 16, kernel_size=64, stride=8)`,
  `ReLU`, `MaxPool1d(4)`, `Conv1d(16, 32, kernel_size=16, stride=2)`, `ReLU`,
  `AdaptiveMaxPool1d(1)`, `Flatten`, `Linear(32, 10)`. Input `(windows, 1, 2048)`, output
  `(windows, 10)`;
- `Rt`, `Rt_217`: the windows of `raw` and `raw_217` divided by the standard deviation of the
  **train** windows of `raw`, float32 tensors of shape `(windows, 1, 2048)` on `DEVICE`;
- `cnn`: `make_cnn()` with `torch.manual_seed(0)` before it, trained with
  `fit_early(..., max_epochs=200)` on `Rt`;
- `cnn_f1`: its test macro-F1.

In [ ]:
def make_cnn():
    ...


Rt, Rt_217 = ..., ...
cnn = ...
cnn_f1 = ...

In [ ]:
# Tests for Task 10 (public). They check what your code does, not how it is written.
# Run this cell after your code; an AssertionError says which check failed.


def test_shapes_and_weights():
    torch.manual_seed(0)
    m = make_cnn()
    out = m(torch.zeros(5, 1, 2048))
    n_w = sum(q.numel() for q in m.parameters())
    assert tuple(out.shape) == (5, 10), f"output (5, 10), got {tuple(out.shape)}"
    by_hand = 16 * (64 + 1) + 32 * (16 * 16 + 1) + 32 * 10 + 10
    assert n_w == by_hand == 9594, f"9594 weights, got {n_w}"


test_shapes_and_weights()


def test_cnn_reported():
    assert tuple(Rt.shape) == (len(raw), 1, 2048)
    assert tuple(Rt_217.shape) == (len(raw_217), 1, 2048)
    assert cnn_f1 is not None and 0 < cnn_f1 <= 1


test_cnn_reported()


print("all tests passed")

**Questions** (answer them in `practice_4_answers.md`, Task 10, with the numbers your code printed):

1. Print the best epoch, the training time, and the test macro-F1 and per-state F1 of the CNN.
2. Put three results side by side: the Section 1 network on 16 time features, the Task 9 network on 32 band energies, this CNN on the raw signal. Count the weights of the CNN and of the Task 9 network. Which needed somebody to design the features, and which found its own?
3. Plot two or three of the 16 first-layer filters (`cnn[0].weight`). Do they look like anything? (A short burst at a few kHz would be a strike detector.)

In [ ]:
# your code for the questions: tables, figures, anything you quote in the answers

---

## Task 11. An autoencoder that knows only the healthy bearing, and a fault at another load  
*0.15 points: 0.07 for the code (tests), 0.08 for the answers*

An autoencoder squeezes its input through a narrow layer and has to rebuild it on the other side.
Trained on healthy windows only, it learns the few directions along which healthy band energies
vary; it is a curved version of PCA. A window of another kind does not lie along those directions,
and its **reconstruction error**, the mean squared difference between the input and the output,

$$\text{err}(\mathbf{x}) = \frac{1}{d}\,\lVert \hat{\mathbf{x}} - \mathbf{x} \rVert^2,$$

is large. The alarm threshold is the largest error among the healthy **validation** windows: no
healthy window the autoencoder has not trained on raises an alarm.

**Write**

- `make_autoencoder()`: an `nn.Sequential` `Linear(32, 16)`, `ReLU`, `Linear(16, 4)`, `ReLU`,
  `Linear(4, 16)`, `ReLU`, `Linear(16, 32)`;
- `recon_error(ae, X)`: the error of every row of the tensor `X`, a 1-D NumPy array, without
  tracking gradients;
- `ae`: `make_autoencoder()` with `torch.manual_seed(0)` before it, trained on the rows of `Bt` that
  are **normal and train**, full batch, Adam `lr=1e-3`, 2000 steps, loss the mean squared error;
- `thr`: the largest error of the normal validation windows;
- `alarm_share`: a `pd.Series` indexed by `STATES`, the share of the test windows of each state with
  an error above `thr`;
- `share_217`: the share of the windows of `Bt_217` above `thr`;
- `cnn_217`: a dict `{state: number of windows}`, what the CNN of Task 10 predicts for the windows
  of `Rt_217` (states it never predicts may be left out).

In [ ]:
def make_autoencoder():
    ...


def recon_error(ae, X):
    ...


ae = ...
thr = ...
alarm_share = ...
share_217 = ...
cnn_217 = ...

In [ ]:
# Tests for Task 11 (public). They check what your code does, not how it is written.
# Run this cell after your code; an AssertionError says which check failed.


def test_autoencoder_shape():
    m = make_autoencoder()
    assert tuple(m(torch.zeros(5, 32)).shape) == (5, 32), "32 in, 32 out"
    n_weights = sum(q.numel() for q in m.parameters())
    assert n_weights == 1220, "32*16+16 + 16*4+4 + 4*16+16 + 16*32+32 weights"


test_autoencoder_shape()


def test_error_on_known_networks():
    X = torch.tensor([[1.0, 2.0], [3.0, -1.0]])
    e0 = recon_error(nn.Identity(), X)
    zero_error = isinstance(e0, np.ndarray) and e0.shape == (2,) and np.allclose(e0, 0)
    assert zero_error, "a perfect copy has error 0 in every row"
    zero = nn.Linear(2, 2)
    with torch.no_grad():
        zero.weight.zero_()
        zero.bias.zero_()
    assert np.allclose(recon_error(zero, X), [2.5, 5.0]), (
        "an output of zeros: the mean of the squared inputs of each row")


test_error_on_known_networks()


def test_alarms_reported():
    assert isinstance(alarm_share, pd.Series) and list(alarm_share.index) == STATES
    assert thr is not None and thr > 0 and share_217 is not None and 0 <= share_217 <= 1
    assert isinstance(cnn_217, dict) and sum(cnn_217.values()) == len(raw_217)


test_alarms_reported()


print("all tests passed")

**Questions** (answer them in `practice_4_answers.md`, Task 11, with the numbers your code printed):

1. Give the threshold, and the share of test windows above it for `normal` and for each fault.
2. **The 3 hp recording.** `raw_217` is IR021 at another motor load; no model has seen it. What do the forest and the network of Task 9, the CNN of Task 10 (count the predicted states) and the autoencoder (share above the threshold) say about it? Which models recognise IR021, which see a fault at all, and which call it normal?
3. One paragraph for a maintenance engineer: which of the two tools, "which fault" or "something is wrong", would you trust on a motor running at a load it was never trained on, and why? What can you not check with these data?

In [ ]:
# your code for the questions: tables, figures, anything you quote in the answers